- rn the hand handles bank stuff and displaying cards
- need to have a way to show and check build (have to pull the text_adv stuff into a .py util file)
- i think to make it more challenging, the player should only be able to sell one card per hand, and anything left in their hand is replaced at the end of the turn
    - could also be good to prevent high value card from spawning in early rounds
- goal is to have a powerful, compatible build

In [ ]:
from IPython.display import clear_output, display
import ipywidgets as widgets
import sys
from pathlib import Path
from IPython.display import Audio

sys.path.append(str(Path.cwd().parent))
from utils.build import check_build, get_missing_parts, show_build
from utils.hand import deal_hand, hand_to_widget
from utils.audio import get_mp3

my_hand = deal_hand()
current_build = show_build()
compatibility_output = widgets.Output()

In [ ]:
win_state = False

def on_win():
    build_cost = sum(card.price_usd for card in current_build.children)
    try:
        print("trying to play audio")
        mp3 = get_mp3(build_cost)
        display(Audio(mp3, autoplay=True))
        print("audio played")
    except Exception as e:
        print(f"(Couldn't play audio: {e})")

In [ ]:
def update_build_compatibility(change):
    global win_state
    with compatibility_output:
        clear_output(wait=True)
        win_state = False
        missing_parts = get_missing_parts(current_build.children)
        if missing_parts:
            print("Still needed:")
            for part in missing_parts:
                print(f"- {part}")
        else:
            win_state = True
            print("All required parts are present.")
        print()
        issues = check_build(current_build.children)
        if issues:
            win_state = False
            print(f"Compatibility issues ({len(issues)}):")
            for issue in issues:
                print(f"- {issue}")
        elif current_build.children:
            print("No compatibility issues found among current build parts.")
        else:
            win_state = False
            print("Build is empty.")
        print("_" * 10)

    if True:
        on_win()

# README!

### Goal: 
- **Make a compatible build**

### Rules:
- **Must have enough in bank to add part in your build**
- **Can only sell one part per hand drawn**
- **Can only sell from your hand, not your build**

_____
Limitations
- *Lighting and fans are not checked in compatibility tests*
- *Buying multiple parts is not supported, build testing will just look at the last part added per category*
- *NO idea why the compat message is being printed so many times. It's on my todo list*

In [ ]:
display(hand_to_widget(my_hand, current_build))

## CURRENT BUILD

In [ ]:
current_build.unobserve_all('children')
current_build.observe(update_build_compatibility, names='children')

display(current_build)
display(compatibility_output)

In [ ]:
len(current_build._trait_notifiers.get('children', {}).get('change', []))